# ADR 031 walkthrough — how one glob call routes to storage

The query under study, in rg/fd spelling — *files named `summary*` of type py or md, under three places*:

```
fd --glob 'summary*' -e py -e md  /data  /data/archive/2025  /code
```

vfs spelling:

```python
await fs.glob("summary*", paths=("/data", "/data/archive/2025", "/code"), ext=("py", "md"))
```

The world — three mounts on heterogeneous backends, one nested, and a scope
root that sits *inside* the nested mount (the case that makes routing
interesting):

```
mount table                          stored rows (entry-local coordinates)
O  owner at /                        /data, /code          (bind-point dirs)
B  at /data          (in-memory)     /reports/summary.py
                                     /reports/summary.md
                                     /notes/summary.txt
                                     /archive               (C's bind-point dir)
C  at /data/archive  (sqlite)        /2024/summary.py
                                     /2025/q1/summary.md
D  at /code          (in-memory)     /tools/summary.py
                                     /tools/summary.json
```

Everything here **runs against the live tree**: the query on both its arms,
the gate, composition, residuation, the recorded dispatches, and the recorded
SQL. The first execution of this walkthrough (2026-08-04) surfaced a live
routing gap — a subsumption-dropped root assertion — which was fixed in the
interim the same day; the notebook now records the fixed behavior and
narrates that history where it happened. The only illustrated (not yet
implemented) part is the final seam shape — the batched `patterns` tuple of
[ADR 031](../../../decisions/031-pattern-only-glob-seam.md) (status:
proposed) — and it is printed from the *computed* routing above it, not
hand-written.

## Step 0 — build the world

Mounts are added through the router (`add_mount` mints the bind-point
directory in the parent entry); rows are written through the router in
namespace coordinates and land in whichever entry owns them. C is a real
`DatabaseStorage` on sqlite so we can record its SQL later.

In [1]:
import tempfile

from vfs.base import VirtualFileSystem
from vfs.storage.backends.database import DatabaseStorage
from vfs.storage.backends.memory import InMemoryStorage

workdir = tempfile.TemporaryDirectory()
fs = VirtualFileSystem()
storage_b = InMemoryStorage()
storage_c = DatabaseStorage(url=f"sqlite+aiosqlite:///{workdir.name}/c.db")
storage_d = InMemoryStorage()

await fs.add_mount(storage_b, "/data")
await fs.add_mount(storage_c, "/data/archive")
await fs.add_mount(storage_d, "/code")

ROWS = (
    "/data/reports/summary.py",
    "/data/reports/summary.md",
    "/data/notes/summary.txt",
    "/data/archive/2024/summary.py",
    "/data/archive/2025/q1/summary.md",
    "/code/tools/summary.py",
    "/code/tools/summary.json",
)
for path in ROWS:
    written = await fs.write(path=path, content="demo", parents=True)
    assert written.success, written.errors
print(f"world built: {len(ROWS)} rows across O(/), B(/data), C(/data/archive), D(/code)")

world built: 7 rows across O(/), B(/data), C(/data/archive), D(/code)


## The query, live

Before dissecting the routing, run the real thing. Expected: the four
py/md rows under the three roots, plus nothing else — `summary.txt` and
`summary.json` are dropped by `ext`, and `/data/archive/2024/summary.py`
*is* included because the root `/data` covers the whole nested mount.

In [2]:
ROOTS = ("/data", "/data/archive/2025", "/code")
result = await fs.glob("summary*", paths=ROOTS, ext=("py", "md"))
print("success:", result.success)
for p in sorted(result.paths):
    print("  ", p)
assert sorted(result.paths) == [
    "/code/tools/summary.py",
    "/data/archive/2024/summary.py",
    "/data/archive/2025/q1/summary.md",
    "/data/reports/summary.md",
    "/data/reports/summary.py",
]

success: True
   /code/tools/summary.py
   /data/archive/2024/summary.py
   /data/archive/2025/q1/summary.md
   /data/reports/summary.md
   /data/reports/summary.py


## Step 1 — gate

`glob_defect` refuses the two silent false friends before any routing:
`**` inside a component, and empty components. The `ext` tuple normalizes
to a case-folded set — one **call-level** fact (caller intent: case
insensitive, empty-extension capable), never per-pattern.

In [3]:
from vfs.glob_patterns import glob_defect

for pattern in ("summary*", "a**b.py", "/data/"):
    print(f"{pattern!r:12} ->", glob_defect(pattern))

'summary*'   -> None
'a**b.py'    -> '**' inside a component ('a**b.py') — use '**' as a whole path segment
'/data/'     -> empty component — every '/' must separate non-empty segments


## Step 2 — composition (ADR 031 decision 3)

`summary*` is name-arm (slash-free). With no anchor channel left at the
seam, each root must *spell* its scoping as pattern text: the gitignore
float made spatial, `root + /**/ + pattern`. Path-arm patterns compose via
the landed `effective_pattern` rule unchanged.

Note what composition *is*: it turns the name-arm query into a path-arm
query. We will run both arms live below and see that today they take
different routes through the router — the ADR unifies them.

In [4]:
from vfs.glob_patterns import effective_pattern
from vfs.paths import Path


def compose(root: str, pattern: str) -> str:
    """ADR 031 D3: path-arm via the landed rule; name-arm floats spatially."""
    if "/" in pattern:
        return effective_pattern(Path(root), pattern)
    prefix = "" if root == "/" else root
    return f"{prefix}/**/{pattern}"


composed = {root: compose(root, "summary*") for root in ROOTS}
for root, pattern in composed.items():
    print(f"{root:22} ->  {pattern}")

/data                  ->  /data/**/summary*
/data/archive/2025     ->  /data/archive/2025/**/summary*
/code                  ->  /code/**/summary*


## Step 3 — residuation per entry

Each entry derives its members from the composed patterns, in its own
coordinates, using the live `residuals`/`render_residual` functions. Two
routing gates, both from the landed planner:

- a composed pattern reaches an entry when its root **covers** the entry's
  bind path (whole-entry reach), or when the root lives **inside** the
  entry *and this entry is the root's owner* — the deepest binding
  containing it. The owner gate is why B never receives
  `/archive/2025/**/summary*`: rows under `/data/archive` are C's, and
  dispatching B there would scan its shadowed region.
- an **empty residual** is the bind-point row itself — the parent's stored
  directory, never a child dispatch.

Watch C: it ends up with **two patterns** — the residual of `/data`'s
composed pattern, and the inside-root's composed pattern in C coordinates.

In [5]:
from vfs.glob_patterns import render_residual, residuals

ENTRIES = {"B": "/data", "C": "/data/archive", "D": "/code"}


def covers(ancestor: str, descendant: str) -> bool:
    return ancestor == "/" or descendant == ancestor or descendant.startswith(ancestor + "/")


def owner(root: str) -> str:
    """The deepest bind path containing *root* — the entry that owns it."""
    return max((b for b in ENTRIES.values() if covers(b, root)), key=len, default="/")


batch: dict[str, tuple[str, ...]] = {}
for entry, bind in ENTRIES.items():
    patterns: set[str] = set()
    for root, pattern in composed.items():
        if not (covers(root, bind) or owner(root) == bind):
            continue
        for components in residuals(pattern, Path(bind)):
            if components:
                patterns.add(render_residual(components))
    batch[entry] = tuple(sorted(patterns))

for entry, patterns in batch.items():
    print(f"{entry} at {ENTRIES[entry]:14} ->  {patterns}")
assert batch["C"] == ("/**/summary*", "/2025/**/summary*")

B at /data          ->  ('/**/summary*',)
C at /data/archive  ->  ('/**/summary*', '/2025/**/summary*')
D at /code          ->  ('/**/summary*',)


## Step 4, today — the name arm, recorded live

Wrap `_dispatch_entry` and re-run the exact query. Today a name-arm scoped
glob does **not** use the pattern machinery above — it rides the generic
fan-out, and the recording shows the dual channel in the flesh:

- `/data` has a binding beneath it, so it names a *region*: B and the
  nested C dispatch **unscoped**, the name-arm pattern broadcast verbatim
  (coordinate-free, the gitignore float).
- `/code` has nothing beneath it, so it stays a **scoped** dispatch — the
  scoping rides the anchor channel (`paths=('/',)`), the pattern again
  untouched.
- the inside-root `/2025` dispatches scoped too (`paths=('/2025',)`).
  **It didn't always**: as landed on 2026-08-01, subsumption dropped this
  dispatch — C already answers unscoped — and the root's existence
  assertion silently vanished with it. This walkthrough's first execution
  caught that (next cell); the same-day interim fix makes a covered glob
  root dispatch on both arms, with the merge deduping the overlap.

Same rows either way — but notice that scoping lives in *two different
channels* depending on mount-table shape. That dual channel is what ADR
031 retires.

In [6]:
async def recorded_glob(pattern: str, **kwargs):
    """Run fs.glob while recording every storage glob dispatch."""
    calls: list[tuple[str, str | None, tuple[str, ...]]] = []
    original = VirtualFileSystem._dispatch_entry

    async def recording(self, binding, op, **kw):
        if op == "glob":
            anchors = tuple(str(p) for p in kw.get("paths", ()))
            calls.append((str(binding.path), kw.get("pattern"), anchors))
        return await original(self, binding, op, **kw)

    VirtualFileSystem._dispatch_entry = recording
    try:
        return await fs.glob(pattern, **kwargs), calls
    finally:
        VirtualFileSystem._dispatch_entry = original


def show(calls) -> None:
    print(f"{len(calls)} storage dispatches:")
    for bind, pattern, anchors in sorted(calls):
        channel = f"anchor channel, paths={anchors}" if anchors else "pattern channel, unscoped"
        print(f"  entry at {bind:14}  pattern={pattern!r:22}  {channel}")


name_arm, name_calls = await recorded_glob("summary*", paths=ROOTS, ext=("py", "md"))
assert sorted(name_arm.paths) == sorted(result.paths)
show(name_calls)
assert len(name_calls) == 4  # was 3 before the 2026-08-04 both-arms fix
assert ("/data/archive", "summary*", ("/2025",)) in name_calls

4 storage dispatches:
  entry at /code           pattern='summary*'              anchor channel, paths=('/',)
  entry at /data           pattern='summary*'              pattern channel, unscoped
  entry at /data/archive   pattern='summary*'              pattern channel, unscoped
  entry at /data/archive   pattern='summary*'              anchor channel, paths=('/2025',)


### The assertion subsumption used to drop

Scope roots are assertions (the find-operand law): a missing root must be
a loud per-root error. As landed on 2026-08-01, the name arm dropped a
covered root's scoped dispatch — and with it the assertion:
`glob("summary*", paths=("/data", "/data/archive/1999"))` returned
**silent success** for a nonexistent root (recorded in ADR 031's context
section). Since the interim fix, both arms fail loud:

In [7]:
bogus = ("/data", "/data/archive/1999")  # /1999 does not exist

name_gap = await fs.glob("summary*", paths=bogus, ext=("py", "md"))
path_gap = await fs.glob("/**/summary*", paths=bogus, ext=("py", "md"))

print("name arm:", "success" if name_gap.success else "failed", [e.path for e in name_gap.errors])
print("path arm:", "success" if path_gap.success else "failed", [e.path for e in path_gap.errors])
assert name_gap.success is False and name_gap.errors[0].path == "/data/archive/1999"
assert path_gap.success is False and path_gap.errors[0].path == "/data/archive/1999"

name arm: failed ['/data/archive/1999']
path arm: failed ['/data/archive/1999']


The interim fix is harm reduction inside the dispatch machinery. ADR 031
decision 4 closes the class structurally: root assertions move out of
query dispatch into one batched point-read per entry (the probe), so
every root is asserted the same way regardless of arm, mount-table shape,
or subsumption — and grep, which shares this fan-out machinery, inherits
the same law when it lands at Pass C.

## Step 5, today — the path arm, recorded live

Now run the *composed* form of the same query — which is exactly what ADR
031's composition rule would produce from the name-arm call. The path arm
does use the residuation machinery, and the single-`pattern` storage slot
forces one dispatch per residual *and* one per scoped root: **four
dispatches, C twice** — once with the residual (unscoped), once through
the anchor channel with its own effective pattern. C's two arms run as
two transactions against two snapshots, and their overlap row is fetched
twice and deduped at the router merge. At 10k roots this shape is the
review's 4–17× and the session-pool exhaustion.

In [8]:
path_arm, path_calls = await recorded_glob("/**/summary*", paths=ROOTS, ext=("py", "md"))
assert sorted(path_arm.paths) == sorted(result.paths)  # same rows as the name arm
show(path_calls)
assert len(path_calls) == 4
assert [c for c in path_calls if c[0] == "/data/archive" and c[2]] == [
    ("/data/archive", "/2025/**/summary*", ("/2025",))
]

4 storage dispatches:
  entry at /code           pattern='/**/summary*'          anchor channel, paths=('/',)
  entry at /data           pattern='/**/summary*'          pattern channel, unscoped
  entry at /data/archive   pattern='/**/summary*'          pattern channel, unscoped
  entry at /data/archive   pattern='/2025/**/summary*'     anchor channel, paths=('/2025',)


## The SQL today, recorded from C

Attach a statement listener to C's engine and re-run the path-arm query.
C's two dispatches arrive as separate transactions: the residual arm's
whole-entry scan, and the scoped arm's anchor subtree fan (`path = '/2025'
OR path LIKE '/2025/%'`) plus its miss-check point-read for the root
assertion. The caller `ext` rides each SELECT as `ext IN (?, ?)` on the
indexed column.

In [9]:
from sqlalchemy import event

statements: list[str] = []
sync_engine = storage_c._host.engine.sync_engine


@event.listens_for(sync_engine, "before_cursor_execute")
def record(conn, cursor, statement, parameters, context, executemany):
    statements.append(statement)


try:
    sql_run = await fs.glob("/**/summary*", paths=ROOTS, ext=("py", "md"))
finally:
    event.remove(sync_engine, "before_cursor_execute", record)
assert sorted(sql_run.paths) == sorted(result.paths)

selects = [s for s in statements if s.lstrip().upper().startswith("SELECT")]
print(f"{len(selects)} SELECTs against C during the one router call:\n")
for statement in selects:
    print(statement)
    print("-" * 72)

3 SELECTs against C during the one router call:

SELECT vfs.content_hash, vfs.created_at, vfs.kind, vfs.mime_type, vfs.path, vfs.size_bytes, vfs.updated_at, vfs.version 
FROM vfs 
WHERE vfs.path != ? AND vfs.path LIKE ? ESCAPE '\' AND vfs.ext IN (?, ?) AND vfs.path != ? AND vfs.path NOT LIKE ? ESCAPE '\'
------------------------------------------------------------------------
SELECT vfs.kind, vfs.path 
FROM vfs 
WHERE vfs.path IN (?)
------------------------------------------------------------------------
SELECT vfs.content_hash, vfs.created_at, vfs.kind, vfs.mime_type, vfs.path, vfs.size_bytes, vfs.updated_at, vfs.version 
FROM vfs 
WHERE vfs.path != ? AND vfs.path LIKE ? ESCAPE '\' AND vfs.ext IN (?, ?) AND vfs.path != ? AND vfs.path NOT LIKE ? ESCAPE '\' AND (vfs.path = ? OR vfs.path LIKE ? ESCAPE '\')
------------------------------------------------------------------------


## Step 6, target — the ADR 031 seam (illustrated)

One call per entry, carrying every live pattern for that entry, executed
in **one transaction, one snapshot**. The calls below are printed from the
`batch` dict computed by real residuation in step 3 — three calls where
the path arm today makes four dispatches, and C's two arms can never tear
or duplicate. The name arm reaches this same seam through composition, so
the dual channel disappears with the channel.

In [10]:
from vfs.glob_patterns import derive_ext

print("ADR 031 target seam — one call per entry, one transaction each:\n")
for entry, patterns in batch.items():
    print(f"  {entry}.glob(patterns={patterns!r}, ext=('py', 'md'), ...)")

print("\nplus the probe: point-read /2025 against C, root rows matched by the router")

print("\nper-arm derived ext (a dotless tail derives nothing):")
print("  derive_ext('summary*')    ->", derive_ext("summary*"))
print("  derive_ext('summary*.py') ->", derive_ext("summary*.py"))

ADR 031 target seam — one call per entry, one transaction each:

  B.glob(patterns=('/**/summary*',), ext=('py', 'md'), ...)
  C.glob(patterns=('/**/summary*', '/2025/**/summary*'), ext=('py', 'md'), ...)
  D.glob(patterns=('/**/summary*',), ext=('py', 'md'), ...)

plus the probe: point-read /2025 against C, root rows matched by the router

per-arm derived ext (a dotless tail derives nothing):
  derive_ext('summary*')    -> None
  derive_ext('summary*.py') -> ('py', '.py')


The statement shape inside C's one call — each pattern is an OR-arm
(literal prefix fans sargably, superset LIKE prefilters, regex is the
authority in Python), the caller `ext` is one call-level conjunct, and a
derived ext (none here — dotless tail) would sit *inside* its arm:

```sql
SELECT ... FROM entry
WHERE ext IN ('md', 'py')                 -- caller ext: once per statement
  AND (
        path LIKE '%/summary%'            -- arm 1: /**/summary*  (superset LIKE)
     OR (path LIKE '/2025/%'              -- arm 2: literal prefix fans the subtree
         AND path LIKE '/2025/%/summary%')
  )
```

With many roots into one entry, the patterns tuple grows instead of the
dispatch count: arms chunk by the bind/OR-depth budgets into several
statements *within* the one transaction — the same posture as every other
batch surface.

| | today, name arm | today, path arm | ADR 031 target |
|---|---|---|---|
| dispatches for this query | 4 since the both-arms fix (3 as landed) | 4 (C twice) | 3 batched calls + probe point-reads |
| scoping channel | anchor channel *and* broadcast pattern (dual) | per-root effective patterns + anchor rels | pattern text only |
| covered root's assertion | dropped as landed; loud since 2026-08-04 | loud | probe — uniform and structural |
| C's snapshot | torn across two transactions | torn across two transactions | single |
| the `/2025/q1/summary.md` overlap | fetched twice, deduped at router merge | fetched twice, deduped at router merge | one candidate, never duplicated |

In [11]:
await storage_c.close()
workdir.cleanup()
print("closed")

closed
